In [3]:
###ONE VARIALE: JOSES DATA_Feb 02 2026

import os
import time
import arcpy

# -------------------------------------------------
# USER SETTINGS
# -------------------------------------------------
root_dir   = r"E:\Data10m"
basins_fc  = r"E:\Polygon_basins\Gruppering.shp"     # <-- your shapefile
basin_id_field = "RegineEnhe"                         # matches folder names like "112a"

raster_name = "spi"                               # finds aspect.tif
out_clipped_dir = r"E:\Mosaic_Norge\clipped_spi_test"  # clipped tiles output
out_mosaic = r"E:\Mosaic_Norge\spi_mosaic_test.tif"    # final mosaic output

pixel_type = "32_BIT_FLOAT"  # your aspect is 32-bit float
mosaic_method = "LAST"       # safe for aspect
# -------------------------------------------------

arcpy.env.overwriteOutput = True
os.makedirs(out_clipped_dir, exist_ok=True)
os.makedirs(os.path.dirname(out_mosaic), exist_ok=True)

def log(msg):
    print(msg)
    try:
        arcpy.AddMessage(msg)
    except:
        pass

def format_eta(seconds):
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def find_raster_in_folder(folder, base_name):
    """Recursively find base_name.tif or base_name.tiff under folder."""
    for dirpath, _, filenames in os.walk(folder):
        for fn in filenames:
            name, ext = os.path.splitext(fn)
            if name.lower() == base_name.lower() and ext.lower() in (".tif", ".tiff"):
                return os.path.join(dirpath, fn)
    return None

# Quick checks
if not arcpy.Exists(basins_fc):
    raise RuntimeError(f"Basins shapefile not found: {basins_fc}")

# Load basin IDs
basin_ids = []
with arcpy.da.SearchCursor(basins_fc, [basin_id_field]) as cur:
    for (bid,) in cur:
        if bid is None:
            continue
        basin_ids.append(str(bid).strip())

basin_ids = sorted(set(basin_ids))
if not basin_ids:
    raise RuntimeError(f"No basin IDs found in field '{basin_id_field}'")

log(f"Found {len(basin_ids)} basin polygons.")

# Make a feature layer for selection
basins_lyr = "basins_lyr"
arcpy.management.MakeFeatureLayer(basins_fc, basins_lyr)

clipped_rasters = []
t_start = time.time()
times = []

for i, bid in enumerate(basin_ids, start=1):
    basin_folder = os.path.join(root_dir, bid)

    if not os.path.isdir(basin_folder):
        log(f"[{i}/{len(basin_ids)}] SKIP (no folder): {basin_folder}")
        continue

    in_raster = find_raster_in_folder(basin_folder, raster_name)
    if not in_raster:
        log(f"[{i}/{len(basin_ids)}] SKIP (no {raster_name}.tif found): {basin_folder}")
        continue

    out_clip = os.path.join(out_clipped_dir, f"{bid}_{raster_name}.tif")
    if arcpy.Exists(out_clip):
        clipped_rasters.append(out_clip)
        log(f"[{i}/{len(basin_ids)}] EXISTS: {out_clip}")
        continue

    # Select polygon for this basin id
    where = f"{arcpy.AddFieldDelimiters(basins_fc, basin_id_field)} = '{bid}'"
    arcpy.management.SelectLayerByAttribute(basins_lyr, "NEW_SELECTION", where)

    sel_count = int(arcpy.management.GetCount(basins_lyr)[0])
    if sel_count != 1:
        log(f"[{i}/{len(basin_ids)}] SKIP (selected {sel_count} polygons for {bid})")
        continue

    t0 = time.time()

    # Clip using polygon geometry
    arcpy.management.Clip(
        in_raster=in_raster,
        rectangle="#",
        out_raster=out_clip,
        in_template_dataset=basins_lyr,
        nodata_value="",
        clipping_geometry="ClippingGeometry",
        maintain_clipping_extent="NO_MAINTAIN_EXTENT"
    )

    dt = time.time() - t0
    times.append(dt)
    clipped_rasters.append(out_clip)

    avg = sum(times) / len(times)
    remaining = (len(basin_ids) - i) * avg
    log(f"[{i}/{len(basin_ids)}] CLIPPED {bid} | {dt:.1f}s | ETA: {format_eta(remaining)}")

# Clear selection
arcpy.management.SelectLayerByAttribute(basins_lyr, "CLEAR_SELECTION")

if not clipped_rasters:
    raise RuntimeError("No clipped rasters were produced. Check folder names and basin IDs.")

log(f"Clipped rasters ready: {len(clipped_rasters)}")
log("Starting final mosaic...")

# Use first clipped raster to set mosaic properties
d0 = arcpy.Describe(clipped_rasters[0])
spatial_ref = d0.spatialReference
band_count = getattr(d0, "bandCount", 1)
cellsize = d0.meanCellWidth

t_m0 = time.time()
arcpy.management.MosaicToNewRaster(
    input_rasters=";".join(clipped_rasters),
    output_location=os.path.dirname(out_mosaic),
    raster_dataset_name_with_extension=os.path.basename(out_mosaic),
    coordinate_system_for_the_raster=spatial_ref,
    pixel_type=pixel_type,
    cellsize=cellsize,
    number_of_bands=band_count,
    mosaic_method=mosaic_method,
    mosaic_colormap_mode="FIRST"
)
t_m = time.time() - t_m0

# Stats/pyramids for good display
log("Calculating statistics + building pyramids...")
arcpy.management.CalculateStatistics(out_mosaic, x_skip_factor=1, y_skip_factor=1, skip_existing="OVERWRITE")
arcpy.management.BuildPyramids(out_mosaic)

total = time.time() - t_start
log("===================================")
log("DONE")
log(f"Clipped rasters : {len(clipped_rasters)}")
log(f"Mosaic time     : {t_m/60:.2f} minutes")
log(f"Total runtime   : {total/60:.2f} minutes")
log(f"Output mosaic   : {out_mosaic}")
log("Clipped folder  : " + out_clipped_dir)
log("===================================")




Found 132 basin polygons.
[1/132] CLIPPED 001a | 19.4s | ETA: 42m 27s
[2/132] CLIPPED 002a | 11.1s | ETA: 33m 2s
[3/132] CLIPPED 002b | 13.0s | ETA: 31m 12s
[4/132] CLIPPED 002c | 20.6s | ETA: 34m 11s
[5/132] CLIPPED 002d | 22.9s | ETA: 36m 48s
[6/132] CLIPPED 002e | 16.0s | ETA: 36m 2s
[7/132] CLIPPED 002f | 12.6s | ETA: 34m 23s
[8/132] CLIPPED 002g | 13.0s | ETA: 33m 13s
[9/132] CLIPPED 002h | 16.2s | ETA: 32m 58s
[10/132] CLIPPED 002i | 12.6s | ETA: 31m 59s
[11/132] CLIPPED 002j | 14.4s | ETA: 31m 29s
[12/132] CLIPPED 002k | 25.8s | ETA: 32m 55s
[13/132] CLIPPED 002l | 25.3s | ETA: 33m 59s
[14/132] CLIPPED 002m | 15.2s | ETA: 33m 26s
[15/132] CLIPPED 002n | 16.7s | ETA: 33m 6s
[16/132] CLIPPED 002o | 18.8s | ETA: 33m 2s
[17/132] CLIPPED 002p | 15.6s | ETA: 32m 35s
[18/132] CLIPPED 002q | 16.2s | ETA: 32m 13s
[19/132] CLIPPED 003a | 21.5s | ETA: 32m 23s
[20/132] CLIPPED 012a | 20.7s | ETA: 32m 26s
[21/132] CLIPPED 012b | 21.4s | ETA: 32m 30s
[22/132] CLIPPED 012c | 22.4s | ETA: 32m 3

In [2]:

###Batch test seams dissapear + snap raster__Feb 02 2026

import os
import time
import arcpy

# -------------------------------------------------
# USER SETTINGS
# -------------------------------------------------
root_dir   = r"E:\Data10m"
basins_fc  = r"E:\Polygon_basins\Gruppering.shp"
basin_id_field = "RegineEnhe"

raster_names = ["aspect", "curv_plan", "curv_prof", "curv_tot"]

out_root = r"E:\Mosaic_Norge\Batch_safe"

pixel_type = "32_BIT_FLOAT"
mosaic_method = "LAST"

NODATA = -9999

reuse_existing_clips = False

# 🔒 SNAP RASTER (ONLY ADDITION)
snap_raster = r"E:\Mosaic_Norge\snap_raster\snap_raster.tif"
# -------------------------------------------------

arcpy.env.overwriteOutput = True
os.makedirs(out_root, exist_ok=True)

def log(msg):
    print(msg)
    try:
        arcpy.AddMessage(msg)
    except:
        pass

def format_eta(seconds):
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

def find_raster_in_folder(folder, base_name):
    for dirpath, _, filenames in os.walk(folder):
        for fn in filenames:
            name, ext = os.path.splitext(fn)
            if name.lower() == base_name.lower() and ext.lower() in (".tif", ".tiff"):
                return os.path.join(dirpath, fn)
    return None

def hard_reset_env():
    arcpy.ResetEnvironments()
    arcpy.env.overwriteOutput = True
    arcpy.env.snapRaster = None
    arcpy.env.extent = None
    arcpy.env.cellSize = None
    arcpy.env.mask = None
    arcpy.env.outputCoordinateSystem = None

def lock_to_snap():
    """Force alignment + extent to snap raster"""
    snap = arcpy.Raster(snap_raster)
    arcpy.env.snapRaster = snap_raster
    arcpy.env.cellSize = snap.meanCellWidth
    arcpy.env.extent = snap.extent

def enforce_nodata(r):
    arcpy.management.SetRasterProperties(r, nodata=f"1 {NODATA}")

# -------------------------------------------------
# Checks
# -------------------------------------------------
if not arcpy.Exists(basins_fc):
    raise RuntimeError(f"Basins shapefile not found: {basins_fc}")

basin_ids = []
with arcpy.da.SearchCursor(basins_fc, [basin_id_field]) as cur:
    for (bid,) in cur:
        if bid is None:
            continue
        basin_ids.append(str(bid).strip())

basin_ids = sorted(set(basin_ids))
log(f"Found {len(basin_ids)} basin polygons.")
log(f"Batch variables: {', '.join(raster_names)}")

t_global0 = time.time()

# -------------------------------------------------
# BATCH LOOP
# -------------------------------------------------
for raster_name in raster_names:
    log("\n" + "="*70)
    log(f"PROCESSING VARIABLE: {raster_name}")

    # 🔥 reset + re-lock snap (THIS IS THE FIX)
    hard_reset_env()
    lock_to_snap()

    out_clipped_dir = os.path.join(out_root, f"clipped_{raster_name}")
    out_mosaic = os.path.join(out_root, f"{raster_name}_mosaic.tif")
    os.makedirs(out_clipped_dir, exist_ok=True)

    basins_lyr = f"basins_lyr_{raster_name}"
    arcpy.management.MakeFeatureLayer(basins_fc, basins_lyr)

    clipped_rasters = []
    t_start = time.time()
    times = []

    for i, bid in enumerate(basin_ids, start=1):
        basin_folder = os.path.join(root_dir, bid)
        if not os.path.isdir(basin_folder):
            continue

        in_raster = find_raster_in_folder(basin_folder, raster_name)
        if not in_raster:
            continue

        out_clip = os.path.join(out_clipped_dir, f"{bid}_{raster_name}.tif")

        if reuse_existing_clips and arcpy.Exists(out_clip):
            enforce_nodata(out_clip)
            clipped_rasters.append(out_clip)
            continue

        where = f"{arcpy.AddFieldDelimiters(basins_fc, basin_id_field)} = '{bid}'"
        arcpy.management.SelectLayerByAttribute(basins_lyr, "NEW_SELECTION", where)

        if int(arcpy.management.GetCount(basins_lyr)[0]) != 1:
            continue

        t0 = time.time()
        arcpy.management.Clip(
            in_raster=in_raster,
            rectangle="#",
            out_raster=out_clip,
            in_template_dataset=basins_lyr,
            nodata_value=str(NODATA),
            clipping_geometry="ClippingGeometry",
            maintain_clipping_extent="NO_MAINTAIN_EXTENT"
        )
        enforce_nodata(out_clip)

        dt = time.time() - t0
        times.append(dt)
        clipped_rasters.append(out_clip)

        avg = sum(times) / len(times)
        remaining = (len(basin_ids) - i) * avg
        log(f"[{raster_name}] [{i}/{len(basin_ids)}] CLIPPED {bid} | {dt:.1f}s | ETA: {format_eta(remaining)}")

    arcpy.management.SelectLayerByAttribute(basins_lyr, "CLEAR_SELECTION")

    if not clipped_rasters:
        continue

    for r in clipped_rasters:
        enforce_nodata(r)

    log(f"[{raster_name}] Starting mosaic...")

    d0 = arcpy.Describe(clipped_rasters[0])
    spatial_ref = d0.spatialReference
    band_count = getattr(d0, "bandCount", 1)
    cellsize = d0.meanCellWidth

    arcpy.management.MosaicToNewRaster(
        input_rasters=";".join(clipped_rasters),
        output_location=os.path.dirname(out_mosaic),
        raster_dataset_name_with_extension=os.path.basename(out_mosaic),
        coordinate_system_for_the_raster=spatial_ref,
        pixel_type=pixel_type,
        cellsize=cellsize,
        number_of_bands=band_count,
        mosaic_method=mosaic_method,
        mosaic_colormap_mode="FIRST"
    )

    enforce_nodata(out_mosaic)
    arcpy.management.CalculateStatistics(out_mosaic, 1, 1, "OVERWRITE")
    arcpy.management.BuildPyramids(out_mosaic)

    log(f"[{raster_name}] DONE")

log("\nALL VARIABLES DONE")
log(f"Total runtime: {(time.time()-t_global0)/60:.2f} minutes")


Found 132 basin polygons.
Batch variables: aspect, curv_plan, curv_prof, curv_tot

PROCESSING VARIABLE: aspect
[aspect] [1/132] CLIPPED 001a | 24.3s | ETA: 53m 1s
[aspect] [2/132] CLIPPED 002a | 13.1s | ETA: 40m 30s
[aspect] [3/132] CLIPPED 002b | 14.6s | ETA: 37m 15s
[aspect] [4/132] CLIPPED 002c | 13.7s | ETA: 35m 2s
[aspect] [5/132] CLIPPED 002d | 15.1s | ETA: 34m 12s
[aspect] [6/132] CLIPPED 002e | 14.1s | ETA: 33m 13s
[aspect] [7/132] CLIPPED 002f | 14.0s | ETA: 32m 24s
[aspect] [8/132] CLIPPED 002g | 14.8s | ETA: 31m 57s
[aspect] [9/132] CLIPPED 002h | 18.2s | ETA: 32m 19s
[aspect] [10/132] CLIPPED 002i | 15.3s | ETA: 31m 57s
[aspect] [11/132] CLIPPED 002j | 13.5s | ETA: 31m 18s
[aspect] [12/132] CLIPPED 002k | 17.5s | ETA: 31m 22s
[aspect] [13/132] CLIPPED 002l | 16.5s | ETA: 31m 14s
[aspect] [14/132] CLIPPED 002m | 9.9s | ETA: 30m 9s
[aspect] [15/132] CLIPPED 002n | 11.4s | ETA: 29m 23s
[aspect] [16/132] CLIPPED 002o | 12.3s | ETA: 28m 47s
[aspect] [17/132] CLIPPED 002p | 9.8s 

In [7]:
####Check alignment

import arcpy

def raster_signature(r):
    d = arcpy.Describe(r)
    ext = d.extent
    return (
        d.spatialReference.factoryCode,
        round(d.meanCellWidth, 6),
        round(d.meanCellHeight, 6),
        round(ext.XMin, 6),
        round(ext.YMax, 6)
    )

rasters = [
    r"E:\Mosaic_Norge\Batch_safe\dtm_mosaic.tif",
    r"E:\Mosaic_Norge\Batch_safe\slope_mosaic.tif",
    r"E:\Mosaic_Norge\Batch_safe\spi_mosaic.tif",
    r"E:\Mosaic_Norge\Batch_safe\tri_mosaic.tif",
    r"E:\Mosaic_Norge\Batch_safe\curv_plan_mosaic.tif",
    r"E:\Mosaic_Norge\Batch_safe\curv_prof_mosaic.tif",
    r"E:\Mosaic_Norge\Batch_safe\aspect_mosaic.tif",
]

for r in rasters:
    print(r, raster_signature(r))



E:\Mosaic_Norge\Batch_safe\dtm_mosaic.tif (25833, 10.0, 10.0, -79745.000009, 7942295.00005)
E:\Mosaic_Norge\Batch_safe\slope_mosaic.tif (25833, 10.0, 10.0, -79745.000009, 7942295.00005)
E:\Mosaic_Norge\Batch_safe\spi_mosaic.tif (25833, 10.0, 10.0, -79745.000009, 7942295.00005)
E:\Mosaic_Norge\Batch_safe\tri_mosaic.tif (25833, 10.0, 10.0, -79745.000009, 7942295.00005)
E:\Mosaic_Norge\Batch_safe\curv_plan_mosaic.tif (25833, 10.0, 10.0, -79745.000009, 7942295.00005)
E:\Mosaic_Norge\Batch_safe\curv_prof_mosaic.tif (25833, 10.0, 10.0, -79745.000009, 7942295.00005)
E:\Mosaic_Norge\Batch_safe\aspect_mosaic.tif (25833, 10.0, 10.0, -79745.000009, 7942295.00005)


In [9]:
####Check alignment per pixel

import arcpy

r1 = arcpy.Raster(r"E:\Mosaic_Norge\Batch_safe\dtm_mosaic.tif")
r2 = arcpy.Raster(r"E:\Mosaic_Norge\Batch_safe\curv_plan_mosaic.tif")

# sample center pixel
row = r1.height // 2
col = r1.width // 2

p1 = arcpy.Point()
r1.extent.lowerLeft.X + col * r1.meanCellWidth
r1.extent.upperLeft.Y - row * r1.meanCellHeight

print("Same grid:", r1.extent == r2.extent and r1.meanCellWidth == r2.meanCellWidth)


Same grid: True


In [10]:

###Remove polygon artifact

import arcpy
from arcpy.sa import Raster, SetNull, IsNull

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

mask = r"E:\Mosaic_Norge\snap_raster\dtm_mask_srast.tif"
wedge_shp = r"E:\Mosaic_Norge\Polygon_ed_mask\Pole_ed_mask.shp"
out_fixed = r"E:\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif"

# Use scratch geodatabase for intermediates (reliable)
scratch_gdb = arcpy.env.scratchGDB
wedge_buf = scratch_gdb + r"\wedge_buf"
wedge_ras = scratch_gdb + r"\wedge_ras"

# Match the mask grid EXACTLY
arcpy.env.snapRaster = mask
arcpy.env.extent = mask
arcpy.env.cellSize = mask
arcpy.env.outputCoordinateSystem = arcpy.Describe(mask).spatialReference

# --- 1) Buffer polygon slightly (optional but recommended)
# If you don't want buffering, comment these two lines and set wedge_to_use = wedge_shp
arcpy.analysis.Buffer(wedge_shp, wedge_buf, "20 Meters")
wedge_to_use = wedge_buf

# --- 2) Polygon -> Raster (burn wedge footprint onto the mask grid)
arcpy.conversion.PolygonToRaster(
    in_features=wedge_to_use,
    value_field="OBJECTID",
    out_rasterdataset=wedge_ras,
    cell_assignment="CELL_CENTER",
    priority_field="NONE",
    cellsize=mask
)

# --- 3) Set mask to NoData where wedge raster has data
m = Raster(mask)
wr = Raster(wedge_ras)
fixed = SetNull(~IsNull(wr), m)

# --- 4) Force GeoTIFF output
arcpy.management.CopyRaster(
    in_raster=fixed,
    out_rasterdataset=out_fixed,
    format="TIFF"
)

print("✅ Wedge removed. Wrote GeoTIFF:", out_fixed)




✅ Wedge removed. Wrote GeoTIFF: E:\Mosaic_Norge\snap_raster\dtm_mask_srast_fixed.tif
